In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report

In [ ]:
#load data
df = pd.read_csv("../data/telco.csv")
#preview
df.head

In [ ]:
df.info #showsData types Missing values, Object vs numeric columns

In [ ]:
df.describe() #show statistical summary.

In [ ]:
df.columns

In [ ]:
df["Churn Label"].value_counts()

In [ ]:
df["Churn Label"].value_counts(normalize=True) #percentage of churned vs active customers.

In [ ]:
df.isnull().sum()

In [ ]:
df['Internet Type'] = df['Internet Type'].fillna('No Internet')
df['Offer'] = df['Offer'].fillna('No Offer')

In [ ]:
# Encode binary categorical features (Yes/No → 1/0)
binary_cols = [
    'Phone Service', 'Paperless Billing', 'Referred a Friend', 'Multiple Lines', 'Online Security', 'Online Backup', 'Device Protection Plan', 'Premium Tech Support', 'Streaming TV', 'Streaming Movies', 'Streaming Music', 'Unlimited Data' 
]
for col in binary_cols:
    df[col] = df[col].map({'Yes': 1, 'No': 0})

In [ ]:
#Encode multi-class categorical features using one-hot encoding
multi_cols = ['Gender', 'Internet Service', 'Internet Type', 'Contract', 'Payment Method']
df = pd.get_dummies(df, columns=multi_cols, drop_first=True)

In [ ]:
#Encode target variable
df.rename(columns={'Churn Label': 'Churn'}, inplace= True)
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

In [ ]:
#check final dataset
print("Shape of dataset:", df.shape)
print("Missing values per column: \n", df.isnull().sum())
print("Target distribution:\n", df['Churn'].value_counts(normalize=True))

In [ ]:
#Define target (y) and features (X)
# Target: 1 = churned, 0 = stayed/joined
y = df["Customer Status"].map({
    "Stayed": 0,
    "Joined": 0,
    "Churned": 1
})

# Drop columns that leak information or are IDs
X = df.drop([
    'Customer Status', 'Churn Label', 'Churn Score', 
    'Churn Category', 'Churn Reason', 'Customer ID',
    'Country','State','City','Zip Code','Latitude','Longitude','Quarter'
], axis=1)

In [ ]:
#Split train/test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [ ]:
#Encode categorical variables
# Convert all object columns to dummy variables
X_train = pd.get_dummies(X_train, drop_first=True)
X_test = pd.get_dummies(X_test, drop_first=True)

# Align columns to avoid missing columns in test set
X_train, X_test = X_train.align(X_test, join='left', axis=1, fill_value=0)

In [ ]:
#Scale numeric features
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
#Train a model
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train_scaled, y_train)

In [ ]:
#Evaluate with confusion matrix
y_pred = model.predict(X_test_scaled)

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:\n", cm)

# Optional: visual display
ConfusionMatrixDisplay.from_estimator(model, X_test_scaled, y_test)

# Detailed metrics
print("\nClassification Report:\n", classification_report(y_test, y_pred))

In [ ]:
#TN = 1026 → Correctly predicted stayed customer
#FP = 9 → Predicted churn but customer stayed
#FN = 59 → Predicted stay but customer 
#TP = 315 → Correctly predicted churned customers

In [ ]:
#Check for potential cheating Optional, check correlation of features with target:

corrs = pd.DataFrame(X_train).corrwith(y_train)
print(corrs[abs(corrs) > 0.9])

In [ ]:
importance = pd.Series(model.feature_importances_, index=X_train.columns) #connects the importance scores from the model to the feature names.

#Sort Features by Importance in ascending order top 10
top_features = importance.sort_values(ascending=False).head(10)

print("Top features influencing churn prediction:")
print(top_features)

In [ ]:
import matplotlib.pyplot as plt

top_features.sort_values().plot(kind='barh')

plt.title("Top Features Influencing Churn")
plt.xlabel("Importance Score")
plt.show()

In [ ]:
#Generate Churn Probabilities
#churn_probabilities = model.predict_proba(X_train.values)[:,1]

In [ ]:
# Make sure to encode all categorical features the same way
X_full = pd.get_dummies(df.drop(columns=['Churn Label']))  # drop target if included

# Align columns with training set
X_full = X_full.reindex(columns=X_train.columns, fill_value=0)

# Predict probabilities for all customers
churn_probabilities = model.predict_proba(X_full.values)[:, 1]

# Assign to the full dataframe
df['Churn_Risk'] = churn_probabilities

In [ ]:
high_risk_customers = df.sort_values(by='Churn_Risk', ascending=False).head(10)

print(high_risk_customers[['Customer ID', 'Churn_Risk', 'Contract', 'Tenure in Months', 'Monthly Charge']])

In [ ]:
import seaborn as sns

sns.histplot(df['Churn_Risk'], bins=30)

plt.title("Distribution of Customer Churn Risk")
plt.xlabel("Churn Probability")
plt.ylabel("Number of Customers")

plt.show()